In [13]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [8]:
SOURCE_DIR = Path("/workspaces/Arch_PC_Assistent/sources/raw_sources")
all_filepaths = list(SOURCE_DIR.rglob("*.txt"))
print(f"found files: {len(all_filepaths)}\n")
for path in all_filepaths[:3]:
    print(f"- {path}")
    
    # So kommen wir später an die Metadaten (Ordnername und Dateiname):
    print(f"  -> Thema (Ordner): {path.parent.name}")
    print(f"  -> Datei: {path.name}\n")

found files: 92

- /workspaces/Arch_PC_Assistent/sources/raw_sources/raw_Arch_source/core/Installation_guide.txt
  -> Thema (Ordner): core
  -> Datei: Installation_guide.txt

- /workspaces/Arch_PC_Assistent/sources/raw_sources/raw_Arch_source/core/General_recommendations.txt
  -> Thema (Ordner): core
  -> Datei: General_recommendations.txt

- /workspaces/Arch_PC_Assistent/sources/raw_sources/raw_Arch_source/core/Users_and_groups.txt
  -> Thema (Ordner): core
  -> Datei: Users_and_groups.txt



In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " "] # Die Reihenfolge, wie er versucht zu schneiden
)

all_chunks = []

print("Starte robusten Chunking-Prozess für alle Dateien...\n")

for filepath in all_filepaths:
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
        
    topic = filepath.parent.name
    filename = filepath.name

    chunks = text_splitter.create_documents([text])
    
    for chunk in chunks:
        chunk.metadata["topic"] = topic
        chunk.metadata["source_file"] = filename
        all_chunks.append(chunk)

print(f"Results: {len(all_chunks)} chunks was succesfully generated from {len(all_filepaths)} files.\n")

Starte robusten Chunking-Prozess für alle Dateien...

Results: 2249 was chunks succesfully generated from 92 files.



In [16]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
persist_directory = "/workspaces/Arch_PC_Assistent/embed_model"

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)


test_query = "How to install the GRUB bootloader?"
results = vectorstore.similarity_search(test_query, k=2)

for i, doc in enumerate(results):
    print(f"--- Hit {i+1} ---")
    print(f"Metadata: {doc.metadata}")
    print(f"Abstract: {doc.page_content[:250]}...\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- Hit 1 ---
Metadata: {'source_file': 'GRUB.txt', 'topic': 'bootloader'}
Abstract: grub-install
requires--no-nvram
or else it errors withefibootmgr: option requires an argument -- 'd'
. This is a long open bug (Ubuntu discussion). The fastest workaround is to manually add the entry for each partition in the array with efibootmgr (G...

--- Hit 2 ---
Metadata: {'source_file': 'GRUB.txt', 'topic': 'bootloader'}
Abstract: grub-install
requires--no-nvram
or else it errors withefibootmgr: option requires an argument -- 'd'
. This is a long open bug (Ubuntu discussion). The fastest workaround is to manually add the entry for each partition in the array with efibootmgr (G...

